# Notebook 02: Data Preprocessing

## Objective
Prepare data for modeling by creating train/test splits.

## Scope
- Load raw data
- Separate features and target variable
- Train/test split (80/20, stratified)
- Save raw split data for reuse in clustering and classification

## Preprocessing Strategy
The following preprocessing steps will be applied **during model training** (in Notebooks 03 & 04):
1. **Class Imbalance Handling**: SMOTE (classification only)
2. **Outlier Handling**: RobustScaler
3. **Feature Scaling**: RobustScaler (optional: log1p transform for skewed features)

## Output
- `split_data.pkl`: Raw train/test split saved for use in subsequent notebooks

In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split
import pickle

In [2]:
DATA_PATH = Path('../')/"data"/"raw_data"/"CDC_Diabetes_Dataset.csv"

In [3]:
# Loading Raw Data 
df = pd.read_csv(DATA_PATH)

In [4]:
df.columns

Index(['Diabetes_012', 'HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker',
       'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies',
       'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth',
       'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education',
       'Income'],
      dtype='str')

In [6]:
# Seperate target and feature
x = df.drop('Diabetes_012', axis=1)
y = df['Diabetes_012']

In [7]:
# Split first to avoid leakage (single-call helper for train/val/test)
def split_train_val_test(X, y, test_size=0.2, val_size=0.2, random_state=42):
    # Split into train and temp, then temp into val/test to preserve stratification
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=(test_size + val_size), random_state=random_state, stratify=y
    )
    val_ratio = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=(1 - val_ratio), random_state=random_state, stratify=y_temp
    )
    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_train_val_test(
    x, y, test_size=0.2, val_size=0.2, random_state=42
    )

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Class distribution (train):\n{y_train.value_counts()}")

Train: (152208, 21), Val: (50736, 21), Test: (50736, 21)
Class distribution (train):
Diabetes_012
0.0    128222
2.0     21207
1.0      2779
Name: count, dtype: int64


In [8]:
# Save raw split data for use in clustering and classification
Path('../data/processed_data').mkdir(parents=True, exist_ok=True)

with open('../data/processed_data/split_data.pkl', 'wb') as f:
    pickle.dump({
        'X_train': X_train,
        'X_val': X_val,
        'X_test': X_test,
        'y_train': y_train,
        'y_val': y_val,
        'y_test': y_test
    }, f)
    
print("Split data saved successfully!")

Split data saved successfully!
